In [14]:
import os
import pandas as pd

# Directory containing parquet files
PARQUET_DIR = r"C:\Users\DalyRob\OneDrive - State of Connecticut\Documents\GitHub Repos\ows-validation-engine\database\cc_db_parquet_output"

# Load each parquet file into its own DataFrame
dfs = {}  # dictionary: table_name -> DataFrame

for fname in os.listdir(PARQUET_DIR):
    if fname.endswith(".parquet"):
        table_name = fname.replace(".parquet", "")
        fpath = os.path.join(PARQUET_DIR, fname)

        df = pd.read_parquet(fpath)
        dfs[table_name] = df

        # Also assign a variable dynamically for convenience in notebooks
        globals()[f"df_{table_name}"] = df

        print(f"Loaded {table_name}: {len(df)} rows, {len(df.columns)} columns")

# Optional: list all loaded DataFrames
print("\nDataFrames created:")
for name in dfs:
    print(" - df_" + name)

Loaded cell_value_history: 5530866 rows, 7 columns
Loaded dataset_column: 349 rows, 4 columns
Loaded participant: 45040 rows, 5 columns
Loaded participant_key_mismatch: 18929 rows, 8 columns
Loaded participant_presence_log: 142563 rows, 7 columns
Loaded person: 65215 rows, 14 columns
Loaded sqlite_sequence: 0 rows, 2 columns
Loaded validation_rule: 0 rows, 5 columns
Loaded validation_run: 570 rows, 8 columns
Loaded validation_violation: 175256 rows, 9 columns

DataFrames created:
 - df_cell_value_history
 - df_dataset_column
 - df_participant
 - df_participant_key_mismatch
 - df_participant_presence_log
 - df_person
 - df_sqlite_sequence
 - df_validation_rule
 - df_validation_run
 - df_validation_violation


In [15]:
df_participant.head()

,participant_id,person_id,dataset_name,org,created_timestamp
0,2dc56994-daab-4816-af45-5e41c24ed770,a401f299-2103-440a-a72d-669f5d03abef,portal data,None,2026-03-18 13:03:02
1,0cf03407-71bd-439a-93ee-580c947cf8f8,8117b88c-c54e-4384-a9a2-adecf63f6312,portal data,None,2026-03-18 13:03:02
2,cdc15945-381c-479f-853b-a2c9f424f31c,475b3619-d620-4f79-be1f-8cc435908dfd,portal data,None,2026-03-18 13:03:02
3,3c64c422-304e-40bd-994c-7359514d859b,8bfa7ac7-48a6-4c69-9abd-7b1a752d07f7,portal data,None,2026-03-18 13:03:02
4,10050685-de86-4a48-9d9a-2e9d3bf5327d,0b52e1d4-ee92-46bd-9378-7ba902255bff,portal data,None,2026-03-18 13:03:02


In [16]:
df_participant.loc[df_participant['org'] == 'Charter_Oak_State_College_Foundation']['participant_id'].nunique()

170

In [17]:
df_participant_presence_log.head()

,run_id,participant_id,status,row_number,sheet_name,quarter,timestamp
0,3dc30fe8-b9b2-499f-9d5d-7c0b9d7e0c3f,2dc56994-daab-4816-af45-5e41c24ed770,present,2.0,Report,PY4_Q2,2026-03-18 13:03:26
1,3dc30fe8-b9b2-499f-9d5d-7c0b9d7e0c3f,0cf03407-71bd-439a-93ee-580c947cf8f8,present,4.0,Report,PY4_Q2,2026-03-18 13:03:26
2,3dc30fe8-b9b2-499f-9d5d-7c0b9d7e0c3f,cdc15945-381c-479f-853b-a2c9f424f31c,present,6.0,Report,PY4_Q2,2026-03-18 13:03:26
3,3dc30fe8-b9b2-499f-9d5d-7c0b9d7e0c3f,3c64c422-304e-40bd-994c-7359514d859b,present,7.0,Report,PY4_Q2,2026-03-18 13:03:26
4,3dc30fe8-b9b2-499f-9d5d-7c0b9d7e0c3f,10050685-de86-4a48-9d9a-2e9d3bf5327d,present,8.0,Report,PY4_Q2,2026-03-18 13:03:26


In [18]:
# See if there are the same amount of participants in Q2 and Q3 for Charter Oak. If there are, then we know that PBI is being wonky. If there are not, then we know that the database is the issue.
# Merge participant with presence log on participant_id
merged = df_participant.merge(
    df_participant_presence_log,
    on="participant_id",
    how="left"
)

# Filter to only the organization you want
filtered = merged[merged["org"] == "Charter_Oak_State_College_Foundation"]

# Show basic info
print("Rows after merge:", len(merged))
print("Rows after org filter:", len(filtered))
print("Unique participant_ids:", filtered["participant_id"].nunique())

filtered.head()

Rows after merge: 142563
Rows after org filter: 1482
Unique participant_ids: 170


,participant_id,person_id,dataset_name,org,created_timestamp,run_id,status,row_number,sheet_name,quarter,timestamp
79496,8a4e5fb1-4d97-4eaa-ae6b-b041ca58042c,3c4b003c-9373-4d5c-b3e8-472c514c30b1,training data,Charter_Oak_State_College_Foundation,2026-03-18 13:05:50,379cef0e-6aef-4ab4-87d9-26c5cb4f6ba7,present,2.0,Personal Information,PY4_Q2,2026-03-18 13:05:52
79497,8a4e5fb1-4d97-4eaa-ae6b-b041ca58042c,3c4b003c-9373-4d5c-b3e8-472c514c30b1,training data,Charter_Oak_State_College_Foundation,2026-03-18 13:05:50,ac7becff-02a5-4a71-9ae4-63886e991e8b,present,2.0,Personal Information,PY3_Q2,2026-03-18 13:54:23
79498,8a4e5fb1-4d97-4eaa-ae6b-b041ca58042c,3c4b003c-9373-4d5c-b3e8-472c514c30b1,training data,Charter_Oak_State_College_Foundation,2026-03-18 13:05:50,6b566f25-6cf8-46f3-9064-4010698adb87,missing,NaN,Personal Information,PY2_Q2,2026-03-18 14:35:49
79499,8a4e5fb1-4d97-4eaa-ae6b-b041ca58042c,3c4b003c-9373-4d5c-b3e8-472c514c30b1,training data,Charter_Oak_State_College_Foundation,2026-03-18 13:05:50,f48352d7-0584-407d-919b-41650eb83ae9,present,2.0,Personal Information,PY2_Q4,2026-03-30 14:07:57
79500,8a4e5fb1-4d97-4eaa-ae6b-b041ca58042c,3c4b003c-9373-4d5c-b3e8-472c514c30b1,training data,Charter_Oak_State_College_Foundation,2026-03-18 13:05:50,eebb3c36-8572-43da-890f-0ccdf7d12a1b,present,2.0,Personal Information,PY3_Q1,2026-03-30 14:18:48


In [19]:
filtered['run_id'].value_counts()

run_id
eebb3c36-8572-43da-890f-0ccdf7d12a1b    170
187d5070-77fa-4ec9-9689-e259701f7375    170
db2b9062-7684-4216-8551-e4a8fa5e7f70    170
f8d1777b-9a36-40c6-afd4-4c649311cee3    170
4dc45389-7f6a-43b2-b196-81a7db239d89    170
f48352d7-0584-407d-919b-41650eb83ae9    165
6b566f25-6cf8-46f3-9064-4010698adb87    165
ac7becff-02a5-4a71-9ae4-63886e991e8b    161
379cef0e-6aef-4ab4-87d9-26c5cb4f6ba7    141
Name: count, dtype: int64

In [20]:
filtered.groupby("quarter")["participant_id"].nunique()

quarter
PY2_Q2    165
PY2_Q4    165
PY3_Q1    170
PY3_Q2    161
PY3_Q3    170
PY3_Q4    170
PY4_Q1    170
PY4_Q2    170
Name: participant_id, dtype: int64